# 06 · Multimodal friction analysis
Modality comparison (RQ3), ablation (RQ4), explanations and the RQ2 statistics.

> **Prerequisite:** `python -m gazegrounded.cli run-all` (synthetic demo).
>
> All logic lives in `src/gazegrounded/`; this notebook only calls it. With the synthetic config every number is **SYNTHETIC DEMO DATA — NOT RESEARCH EVIDENCE**.

In [1]:
from pathlib import Path
import os
# Run from the repository root so relative config paths resolve.
while not Path("pyproject.toml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
import numpy as np
import pandas as pd
from gazegrounded.config import load_config
from gazegrounded.data.loader import load_processed
from gazegrounded.utils import SYNTHETIC_BANNER
cfg = load_config("configs/synthetic.yaml")  # switch to configs/experiment.yaml for real data
ds = load_processed(cfg)
print(SYNTHETIC_BANNER if ds.is_synthetic else "Real data:", cfg.data.source)

SYNTHETIC DEMO DATA — NOT RESEARCH EVIDENCE synthetic


In [2]:
from gazegrounded.multimodal.features import MODALITY_FEATURES
feat = pd.read_csv(cfg.processed_dir()/'features.csv')
{m: len(c) for m, c in MODALITY_FEATURES.items()}

{'gaze': 10, 'interaction': 4, 'vlm': 8, 'text': 0, 'visual': 3}

In [3]:
from gazegrounded.utils import latest_run_dir
run = latest_run_dir(cfg.experiments_dir(), 'friction')
res = pd.read_csv(run/'modality_comparison.csv')
res.pivot_table(index='modality', columns=['group_by','model'], values='macro_f1').round(2)

group_by               participant_id                         \
model             logistic_regression majority random_forest   
modality                                                       
gaze                             0.67      NaN          0.66   
interaction                      0.72      NaN          0.69   
majority_baseline                 NaN     0.25           NaN   
multimodal                       0.72      NaN          0.72   
text                             0.64      NaN          0.62   
visual                           0.65      NaN          0.65   
vlm                              0.60      NaN          0.67   

group_by                        ui_id                         
model             logistic_regression majority random_forest  
modality                                                      
gaze                             0.61      NaN          0.63  
interaction                      0.75      NaN          0.67  
majority_baseline                 NaN     0.05           NaN  
multimodal                       0.53      NaN          0.65  
text                             0.56      NaN          0.53  
visual                           0.42      NaN          0.42  
vlm                              0.41      NaN          0.30

Note how VLM and visual features drop when generalising to **new interfaces**: with five screens they partly identify the interface.

In [4]:
pd.read_csv(run/'ablation.csv')[['dropped_modality','macro_f1','quadratic_weighted_kappa']].round(3)

,dropped_modality,macro_f1,quadratic_weighted_kappa
0,gaze,0.721,0.843
1,interaction,0.693,0.849
2,vlm,0.748,0.875
3,text,0.667,0.825


In [5]:
imp = pd.read_csv(run/'feature_importance.csv')
imp[imp.kind=='permutation_macro_f1_drop'].sort_values('importance_mean', ascending=False).head(10).round(3)

,feature,importance_mean,model,kind,class,coefficient,importance
7,fixation_count,0.121,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
5,completion_time_ms,0.080,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
19,text,0.067,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
12,revisit_count,0.053,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
20,ttff_censored_ms,0.042,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
1,aoi_transition_count,0.041,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
10,mean_fixation_duration_ms,0.037,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
16,task_relevant_attention,0.036,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
3,attention_divergence_early,0.035,logistic_regression,permutation_macro_f1_drop,NaN,NaN,NaN
42,text,0.030,random_forest,permutation_macro_f1_drop,NaN,NaN,NaN


## RQ2 statistics (repeated-measures correlation is primary)

In [6]:
from gazegrounded.analysis.research import rq2_statistics
from gazegrounded.analysis.reporting import format_test
for r in rq2_statistics(cfg, feat).query("role == 'primary'").to_dict('records'):
    print(f"{r['x']} vs {r['y']}: {format_test(r)}")

attention_divergence_early vs completion_time_ms: rmcorr, r = 0.50 [0.35, 0.63], p < .001, p_Holm < .001, N = 120
attention_divergence_early vs revisit_count: rmcorr, r = 0.46 [0.30, 0.60], p < .001, p_Holm < .001, N = 120
attention_divergence_early vs ttff_censored_ms: rmcorr, r = 0.53 [0.38, 0.65], p < .001, p_Holm < .001, N = 120
condition vs attention_divergence_early: wilcoxon, W = 0.00, p < .001, p_Holm < .001, rank_biserial = 1.00, N = 12
condition vs completion_time_ms: wilcoxon, W = 0.00, p < .001, p_Holm < .001, rank_biserial = 1.00, N = 12
